# 第2回 演習：回帰モデルを立てる

## 今日の分析目標

**気温から利用台数を予測する式を、自分の手で立てる。**

`fit()` を呪文のまま使うのではなく、「良い直線を残差の二乗和で選ぶ」という中身を自分の手で確かめることが目標。TODOに取り組みながら、最後の「目標に答えられたか」で振り返りましょう。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
import japanize_matplotlib

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
print('セットアップ完了')

## 1. データの読み込み

前回と同じ自転車シェアデータ。今日は気温 `x` と利用台数 `y` の2列を中心に使います。

### 深掘り：予測の式の形——傾き a と切片 b が背負う意味

これから立てるのは「**台数 ≈ a × 気温 + b**」という、いちばん単純な**直線の式**です。関係はもっと複雑かもしれないのに、なぜ最初から直線を選ぶのか。理由は二つあります。ひとつは、直線が「気温が上がれば台数も増える」という素直な傾向を、たった2つの数字で言い切れる**いちばん見通しのよい形**だから。もうひとつは、あとで足りないと分かってから項を足す（2乗の項を入れる、変数を増やす）ほうが、いきなり複雑な式から始めるより安全だからです。まずは直線から、が定石です。

無限にありそうな予測式の形を、この直線を選んだ瞬間に「a・b という2つの数字を決めるだけ」の問題へ絞り込めます。ここで a と b が何を意味するかを、先に言葉にしておきましょう。

- **a（傾き）**：気温が1増えたとき、台数がいくら増えるか。単位は「台 ÷ 気温1」です。このデータの気温は0〜1に正規化されているので、a は「最も寒い日（気温0付近）から最も暑い日（気温1付近）まで動いたときの、台数の変化幅」とほぼ同じ意味になります。
- **b（切片）**：気温が0のときの台数の基準点。直線が縦軸と交わる高さです。

第1回の演習で、利用台数と最も強く関係していたのが気温（相関 **0.63**）でした。関係がプラス向きなので、当てはめる直線の傾き a もプラスになるはず——右上がりの直線を1本引く、というのがこの回の主題です。

問題は、a と b の組み合わせが無数にあること。「点の真ん中を通る感じ」では人によって答えが変わり、他人に説明もできません。そこで、直線の良し悪しを**数字で測る物差し**を用意し、その物差しが最も良くなる一点を選びます。`fit()` に丸投げすれば一行で答えは出ますが、この演習ではその一行の中身——物差しを決めて最小の点を選ぶ、という手続き——を自分の手で組み立てます。次の節でその物差し（**残差の二乗和**）を作ります。

In [ ]:
DATA_DIR = 'https://raw.githubusercontent.com/k0heiun0/applied_exercise/main/shared/data'   # データはこのリポジトリから読み込む
df = pd.read_csv(f'{DATA_DIR}/bike_day.csv')
x = df['temp'].values
y = df['cnt'].values
print(f'日数: {len(x)}')
df[['temp', 'atemp', 'hum', 'cnt']].head(3)

## 2. 直線を1本引いて、二乗和で採点する

まず手始めに、勘で置いた直線 `a=9000, b=0` を散布図に重ね、残差の二乗和（SSE）で採点してみます。

### 深掘り：二乗和という物差しと、谷が一つだけになる理由

上のセルで作った `sse(a, b)` は、各日の予測の外し（残差 = 実際 − 予測）を**二乗して全日ぶん足した**値です。式で書けば

$$
\mathrm{SSE}(a, b) = \sum_{i=1}^{n} \bigl(y_i - (a\,x_i + b)\bigr)^2 .
$$

**なぜ二乗するのか**　残差をそのまま足すと、上に外した日（プラス）と下に外した日（マイナス）が**打ち消し合って**しまいます。たとえば+300台の外しと−300台の外しを足すと0——二回も外したのに「無傷」に見えてしまう。これでは物差しになりません。二乗すれば符号が消えて打ち消しが起きず、しかも大きく外した日ほど（2の二乗は4、10の二乗は100と）**急激に重く**数えられます。「大外れを強く罰する物差し」になっているわけです。

**数字の桁に意味はない、比較にだけ意味がある**　実際に当ててみると、勘で置いた `a=9000, b=0` の SSE は約 **18.0 億**（`1,798,539,710`）、当たりに近い `a=6600, b=1200` は約 **16.6 億**（`1,661,773,086`）でした。18億という値そのものに深い意味はありません——731日ぶんの、しかも台数の二乗を足し込んだ合計なので大きくなって当然です。大事なのは**同じデータの上で比べたときの大小**で、16.6億 < 18.0億 だから後者が良い、と言える。目で見た「当たっている」が数字の大小に翻訳されました。

**谷底が一つだけになる**　この SSE を、a・b を軸にした地形として思い浮かべてください。SSE は a・b について**二次式（下に凸の放物面・お椀の形）**なので、くぼみは**ただ一つ**。あちこちに落とし穴があるのではなく、最も低い一点へ向かってするすると下る形をしています。b を各 a に対して最良の値（直線が平均の点を通る値）に合わせて a だけ動かすと、この地形の断面がきれいな**U字**になります。傾きが小さすぎても大きすぎても二乗和は大きく、真ん中の一点で最小——だから「いちばん良い直線」が迷いなく1つに決まります。総当たりで a を細かく振ってU字の底を探すこともできますが、次の節ではこの底を**式一発**で求めます。谷が一つしかない（凸である）という性質こそ、その近道を支える土台です。

**別の見方とつまずき**　物差しは二乗和が唯一ではありません。残差の**絶対値**を足す測り方（平均絶対誤差）もあり、そちらは大外れを二乗ほど重くは罰しません。二乗和を選ぶと計算が「式一発」になる利点がある反面、**極端な1日（外れ値）に直線が引きずられやすい**という弱点も同時に抱えます。大外れを重く数える長所と、外れ値に弱い短所は裏表だ——物差しを選ぶとは、こうした性質ごと選ぶことだと覚えておきましょう。

In [ ]:
def sse(a, b):
    residual = y - (a * x + b)
    return (residual ** 2).sum()

a0, b0 = 9000, 0
plt.scatter(x, y, s=8, alpha=0.2, color='#0066cc')
xx = np.linspace(x.min(), x.max(), 50)
plt.plot(xx, a0 * xx + b0, color='#e63946', lw=2.5, label=f'a={a0}, b={b0}')
plt.xlabel('temp')
plt.ylabel('cnt')
plt.legend()
plt.show()
print(f'a={a0}, b={b0} の残差二乗和: {sse(a0, b0):,.0f}')

### TODO①：自分で a, b を選んで二乗和を計算

散布図を見ながら、`a=9000, b=0` より当たりそうな a と b を自分で選び、`sse()` で採点してください。何度か値を変えて、二乗和をどこまで小さくできるか試してみましょう。

In [ ]:
# TODO: 自分で a と b を選んで、sse(a, b) を計算してください（9000, 0 より小さくできる？）
# ヒント: 散布図の点の帯は、傾き6600前後・切片1200前後の直線に沿って見える
...

## 3. 最小二乗解を numpy で計算する

いちばん良い a と b（残差二乗和の谷底）は、共分散と分散から式一発で求まります。

### 深掘り：最小二乗解は「式一発」で出る——正規方程式と、射影という見方

上のセルは `a = 共分散 ÷ 分散`、`b = ȳ − a·x̄` という2行で谷底を出しました。これは総当たりでU字の底を探さなくても、最小二乗解が**閉じた式で一発**で求まることを意味します。じつはこの2行は、もっと一般的な公式——**正規方程式**——の、変数が1本のときの特別な場合です。

**正規方程式（骨子）**　変数を何本使っても、最小二乗解 $\hat\beta$ は次の一本の式で書けます。

$$
\hat\beta = (X^{\top} X)^{-1} X^{\top} y .
$$

記号を一語ずつほどきます。

- $X$：**計画行列**。1行が1日、列が使う変数です。先頭に「全部1」の列を1本足しておくと、その列の係数がそのまま切片 b になります。今回の単回帰なら $X$ は「1の列」と「気温の列」の2列、$n=731$ 行。
- $y$：実測した利用台数を縦に並べたベクトル（731個）。
- $\hat\beta$：求めたい係数を並べたベクトル。単回帰なら $(b,\ a)$ の2つ、重回帰なら変数の数だけ並びます。
- $X^{\top} X$ は変数どうしの内積を集めた小さな正方行列、$X^{\top} y$ は各変数と y の内積のベクトル。その逆行列を掛けたものが答えです。

**なぜ「一発」で出るのか**　前節の通り SSE は $\beta$ についての二次式（お椀）なので、谷底は「傾き（勾配）がゼロになる一点」です。SSE を $\beta$ で微分してゼロと置くと、ちょうど $X^{\top} X\,\hat\beta = X^{\top} y$ という連立一次方程式（これが**正規方程式**の名の由来）になり、両辺の左から逆行列を掛ければ上の閉じた式になります。探索も繰り返しもいらず、行列計算だけで**二乗誤差を最小にする一点**が出る——これが「式一発」の中身です。谷が一つ（凸）だからこそ、勾配ゼロの点がそのまま最小になります。

**実データで単回帰の特別な場合を確かめる**　気温1本だけのとき、この公式は上のセルの2行にぴたりと一致します。共分散 **222.21** ÷ 分散 **0.03346** = **6640.71**（= a）、切片は $b = \bar y - a\,\bar x = 4504.35 - 6640.71 \times 0.4954 = 1214.64$。行列版 $(X^{\top}X)^{-1}X^{\top}y$ に入れても、答えは同じ $(b, a) = (1214.64,\ 6640.71)$ です。単回帰の素朴な式と正規方程式は、別物ではなく同じものを見ていた、というわけです。傾き **6640.71** の意味も言葉にしておくと、「気温の指標が0から1へ、つまり最も寒い日から最も暑い日まで上がると、利用台数はおよそ6640台増える」という関係を表しています。

**幾何で見る：最小二乗＝y を列空間へ正射影する**　もう一つの見方が、この公式の腑に落ちる近道です。予測 $\hat y = X\hat\beta$ は、$X$ の列（1の列と気温の列）を混ぜて作れる値の全体——**列空間**——の中の一点です。実測 $y$ はふつうこの空間の外（731次元のどこか）にあります。最小二乗が選ぶのは、列空間の中で**$y$ にいちばん近い点**、すなわち $y$ から列空間へ**まっすぐ下ろした垂線の足（正射影）**です。「二乗誤差の最小化」とは、この「いちばん近い」を距離の二乗で測ることにほかなりません。

このとき残差 $y - \hat y$ は列空間と**直角**になり、$X^{\top}(y - X\hat\beta) = 0$ が成り立ちます——これを整理したものが、まさに正規方程式そのもの。「二乗誤差を最小化する」と「残差を列空間に直角に落とす」が同じことだ、と分かります。この直交性は、上の節の実行結果とも符合します。計画行列には「全部1」の列が入っているので、残差はその列とも直交します。つまり**残差の総和がゼロ**、言い換えれば**残差の平均が0**になる——後の残差セルで平均がぴたりと0になるのは、この射影の直角の性質の現れです。ちなみに、この射影を一手で表す行列 $H = X(X^{\top}X)^{-1}X^{\top}$（$\hat y = Hy$）はハット行列と呼ばれ、後の回で残差の性質を調べるときに顔を出します。

**つまずきどころ：切片 b の意味**　b = **1214.64** は「気温が0のときの基準台数」ですが、これを「気温0の日は約1215台」と真に受けるのは危険です。このデータの気温の最小は **0.059**（最も寒い日でも0より少し大きい）で、気温ちょうど0は**観測の外側**にあります。切片はあくまで直線を縦軸まで伸ばした先の値であって、実際に観測された台数ではない——**式の守備範囲の外**を額面どおり読まない、というのが切片とのつき合い方です。

厳密な導出（SSE を微分してゼロと置く計算、$X^{\top}X$ に逆行列が存在するための条件など）は、**詳しくは MVA『回帰』回**へ。ここでは「二乗誤差の谷底が、正射影として一発で決まる」という筋道をつかめれば十分です。

In [ ]:
a_best = np.cov(x, y, ddof=0)[0, 1] / np.var(x)
b_best = y.mean() - a_best * x.mean()
print(f'numpy の最小二乗解: a = {a_best:.2f}, b = {b_best:.2f}')
print(f'そのときの残差二乗和: {sse(a_best, b_best):,.0f}')

### TODO②：numpy 解と scikit-learn の一致確認

`LinearRegression` を `df[['temp']]` と `y` で学習させ、係数 `coef_` と切片 `intercept_` が上の `a_best`, `b_best` と一致することを print で確認してください。

In [ ]:
# TODO: LinearRegression().fit() を実行し、coef_ と intercept_ を a_best, b_best と並べて print してください
# ヒント: model = LinearRegression() → model.fit(df[['temp']], y) → model.coef_[0] と model.intercept_
...

## 4. 予測と R²（決定係数）

学習済みモデルで予測し、当てはまりの粗い物差し R² を見てみます（R² の詳しい意味は第5回で扱います）。

### 深掘り：係数の符号は読める、大きさは比べられない——単位のわな

式が立つと、代入するだけで予測できます。上のセルの通り、気温0.5の日なら約 **4535 台**。これは「呪文」ではなく、求めた a・b の式に値を入れているだけです。ここから変数を増やした**重回帰**へ進むと、係数の**読み方**につまずきどころが出てきます。

**符号は素直に読める**　11変数の重回帰を当てると、係数はおよそ次のようになります。

| 変数 | 係数 | 符号の読み |
|---|---:|---|
| temp（気温） | +2029 | 暖かいほど増 |
| atemp（体感温度） | +3573 | 暖かいほど増 |
| yr（年） | +2041 | 2年目ほど増 |
| season（季節） | +510 | 番号が進むほど増 |
| weathersit（天気） | −611 | 悪天候ほど減 |
| hum（湿度） | −1019 | 湿るほど減 |
| windspeed（風速） | −2558 | 強風ほど減 |

まず**符号**は素直に読めます。湿度・風速・悪天候はマイナス（増えると利用が減る）、年はプラス（2年目のほうが利用が多い）——直感に合う向きです。「どちら向きに効くか」は、この時点でも十分に語れます。

**つまずき：大きさは直接比べてはいけない**　いっぽう係数の**大きさ**で重要度を比べるのは早合点です。理由は**変数ごとに単位（とれる範囲）が違う**から。temp・atemp・hum・windspeed は0〜1に正規化された値、season は1〜4、mnth は1〜12、yr は0か1——土俵がバラバラです。たとえば係数が大きく見える temp（+2029、動く幅は約0.8）と、係数が小さい mnth（−39、動く幅は11）を、数字の大小だけで「temp のほうが重要」とは言えません。係数は「その変数が**1**増えたときの効き」ですが、その“1”の重み（1がどれだけ大きな変化か）が変数ごとに違うのです。**係数が大きい＝重要**、という読み方は単位のわなにはまっています。単位のそろわない係数を並べて棒グラフにし、長い棒を「効いている変数」と読むのは、この演習で避けたい典型的な早合点です。

**そっくりな2列のふるまい**　単回帰で当てはまりを測ると、気温 temp の R² は **0.394**、体感温度 atemp は **0.398** と**ほぼ同じ**。2列の相関は0.99で、実質そっくりな双子だからです。単独ではほぼ互角なのに、重回帰で両方を入れると係数は temp **+2029** / atemp **+3573** と割れて、いちばん効きそうな気温が体感温度に食われた形になります。同じ情報を持つ2列に「効き」をどう振り分けるかをデータが決めきれず、係数が不安定になる現象で、第6回・第7回の演習でその正体（多重共線性）と対処を扱います。TODO③で `atemp` を選ぶと R² が temp とほぼ同じになるのは、この双子性の現れです。

なお、係数の大きさを公平に比べたいなら、各変数を平均0・ばらつき1に**標準化**してから当てはめ、無単位にそろえる手（標準化係数）があります。土俵をそろえれば「1増えたとき」ではなく「1ばらつきぶん動いたとき」の効きで比べられ、大きさの比較に意味が出ます。この演習では未処理のままなので係数の大きさ比べは保留で、標準化して土俵をそろえる話は第7回の演習で扱います。

最後にもう一点。単回帰の R² **0.394** が、11変数の重回帰で **0.80** まで上がりました。変数を増やすと当てはまりは確かに改善します。ただしどちらも**学習に使ったデータでの自己採点**にすぎず、新しいデータでも同じく当たるかは別問題です（次回のテーマ）。

In [ ]:
model_temp = LinearRegression()
model_temp.fit(df[['temp']], y)
pred = model_temp.predict(pd.DataFrame({'temp': [0.2, 0.5, 0.8]}))
for t, p in zip([0.2, 0.5, 0.8], pred):
    print(f'気温 {t:.1f} → 予測台数 {p:.0f} 台')
r2_temp = model_temp.score(df[['temp']], y)
print(f'temp 単回帰の R2: {r2_temp:.3f}')

### TODO③：別の1変数で単回帰して R² を比較

`temp` 以外の変数（例: `atemp`, `hum`, `windspeed` など）を1つ選んで単回帰し、R² を `temp` の場合と比べてください。気温より当てはまる変数はあるでしょうか？ また `atemp`（体感温度）の結果は `temp` とどう違うでしょうか？

In [ ]:
# TODO: temp 以外の変数を1つ選んで LinearRegression で単回帰し、R2 を r2_temp と比較してください
# ヒント: model.fit(df[['変数名']], y) → model.score(df[['変数名']], y)
...

## 目標に答えられたか

- 今日の目標は「気温から利用台数を予測する式を、自分の手で立てる」ことでした
- TODO①で選んだ a, b の二乗和は、numpy の最小二乗解にどこまで迫れましたか？
- TODO②で、`fit()` の答えと自分の計算は一致しましたか？ 一致した理由を自分の言葉で説明できますか？
- TODO③で、`atemp`（体感温度）の R² は `temp` とほぼ同じになりませんでしたか？ それはなぜでしょう？（→第6回のテーマ）
- ここで測った R² は「学習に使ったデータでの成績」です。新しいデータでも同じくらい当たると言えるでしょうか？（→次回のテーマ）

## 課題（提出）

**提出するもの**: 応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

4節では気温 `temp` 1本で単回帰をしました。今度は説明変数を2本にします。
`temp` と `hum`（湿度）の2列で `LinearRegression` を学習し、2つの係数と切片、そして R² を表示してください。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

4節の `model_temp` と同じ書き方で、`df[['temp']]` を `df[['temp', 'hum']]` に変えるだけです。係数は `.coef_` に2つ並びます。R² は `.score()` で出ます。

</details>


### 応用②（判断）

応用①の R² は、4節の temp 単回帰の R²（`r2_temp`）から**幾つ上がりました**か。差を**小数第3位まで**答えてください（例: 0.012）。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

応用①のモデルの `.score(df[['temp', 'hum']], y)` から `r2_temp` を引き、`round(差, 3)` で丸めます。

</details>


### 応用③（解釈）

気温だけの式に湿度 `hum` を足す価値はありますか。R² の上がり幅と、`hum` の係数の符号（プラスかマイナスか）の2つを根拠に、自転車シェアの運営担当者に向けて3行で書いてください。


（ここに3行程度で書く）


## 発展（任意）

### statsmodels で係数の「信頼度」を読む

scikit-learn の `LinearRegression` は予測のための道具です。係数と R² は出ますが、「その係数は偶然ではないと言えるか」「本当の値はどのくらいの幅にあるか」は教えてくれません。

同じ最小二乗法でも、統計パッケージ `statsmodels` を使うと、係数ごとに **p値**（係数が本当は0なのに、これだけ以上0から離れた値が偶然出る確率）と **95%信頼区間**（本当の係数がありそうな幅）が出ます。

論文や報告書で回帰の結果を示すときは、こちらの表が標準です。応用①と同じ `temp` と `hum` の重回帰を、`statsmodels` で当ててみます。


In [ ]:
# Colab には入っていますが、なければインストールする
import importlib.util
if importlib.util.find_spec('statsmodels') is None:
    %pip install -q statsmodels


In [ ]:
import statsmodels.api as sm

X = sm.add_constant(df[['temp', 'hum']])   # 切片用の「全部1」の列を先頭に足す
ols = sm.OLS(y, X).fit()
print(ols.summary())


**読み方**　表の真ん中のブロックを見ます。`coef` の列が係数で、temp が 6886.9737、hum が −2492.8541、const（切片）が 2657.8951 です。小数第1位に丸めれば応用①の 6887.0、−2492.9、2657.9 で、scikit-learn の値と一致します（同じ最小二乗法なので当然です）。

`P>|t|` の列が p値です。temp も hum も 0.000 と表示されており、「係数が本当は0」とは考えにくい、つまり偶然の効きではないと読めます。

`[0.025  0.975]` の2列が95%信頼区間です。hum は −3248 から −1737 で、区間が **0 をまたいでいません**。マイナスの向きに効くという結論は、この幅で見ても符号は変わりません（幅の前提は残差の独立性など。詳しくは MVA『重回帰』回）。

右上の `R-squared` は 0.427 で、応用①の R² と同じ値です。

3節で見た「全部1の列を足す」が `sm.add_constant` そのもので、`sm.OLS` は同じ最小二乗解を求めています。p値や信頼区間がどう計算されるかは、**詳しくは MVA『重回帰』回**へ。ここでは「係数には幅がある。0をまたぐ区間の係数は、符号すら決めきれない」という読み方をつかめば十分です。

試すなら、`df[['temp', 'hum', 'windspeed']]` のように `windspeed` を足して、3本目の区間も 0 をまたがないか見てください。
